# 01c — Tách và gán nhãn DoS từ TII-SSRC-23 + benign TII làm nền khó

Bổ sung lớp **`dos`** cho pipeline cửa sổ gói (03b → 04b), cùng định dạng đầu ra với notebook 01.

**DoS** (`pcap/malicious/dos/`): một máy tấn công IP thật `192.168.1.70` → nạn nhân `192.168.1.90`, hping3.
Mỗi file là nhiều **đợt** (cách nhau ~5 s) gom thành **phiên** (cách nhau hàng trăm–nghìn giây), tốc độ 10 → ~15.000 gói/s.
Dùng ICMP/SYN/UDP + ACK/FIN/RST flood (`INCLUDE_TCP_FLAG_FLOODS`). Không dùng: CWR/ECN/PSH/URG flood (để dành làm
test ngoài — loại flood chưa thấy khi train) và HTTP flood (ngoài phạm vi, lẫn với brute-force).
Phạm vi lớp `dos`: một máy tấn công IP thật (không phải DDoS giả nguồn).

1. **Đợt**: gói `.70 → .90` cách gói trước > `BURST_GAP_S` thì mở đợt mới; tốc độ = số gói gửi / thời lượng.
2. **Bỏ đợt < `MIN_DOS_PPS`** (10–40 gói/s): chậm hơn cả video UDP benign của TII (~180 gói/s) cùng cặp máy,
   không gây DoS và không thể tách khỏi traffic thường — giữ lại chỉ dạy model báo động traffic thường.
3. **Phiên** = các đợt giữ lại cách nhau ≤ `SESSION_GAP_S` → một **task** (như task lab).
   Gói `attack` = mọi gói IP giữa `.70` và `.90` trong đợt giữ lại (+`RESP_TAIL_S` cho gói trả lời muộn);
   ARP của hai máy trong phiên = `aux`; còn lại bỏ.
4. **Đổi địa chỉ** `.70`/`.90` (IP + MAC) → attacker `.236` / target `.182` của lab, để 03b trộn y như task lab
   (persona client → `.236`, camera → `.182`).
5. **Chia theo thời gian** trong từng loại: phiên sớm → train, phiên muộn → val/test (~70/15/15, mỗi split ≥ 1 phiên
   khi đủ phiên) — mỗi loại chỉ có một file, chia ngẫu nhiên sẽ để các đợt liền nhau rơi vào cả train lẫn test.

**Benign TII** (`pcap/benign/`: video HTTP/RTP/UDP, audio, text, background) — cùng testbed, cùng máy `.70`/`.90`:
nền "khó" (traffic hợp lệ tốc độ cao) để model không báo DoS nhầm. Mỗi file cắt liên tục theo thời gian
70/15/15 → train/val/test, giữ nguyên IP (không trộn, 03b tính cửa sổ trực tiếp).

Đầu ra:
- `Data_after_processing/attack_packets/dos_<loại>/task_XXXX.parquet` (cột giống notebook 01)
- `Data_after_processing/benign_tii/<tên>_<split>.parquet`
- `reports/01c_dos_bursts.csv`, `reports/01c_dos_task_inventory.csv`, `reports/01c_tii_benign_parts.csv`

In [1]:
import csv, subprocess
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

ROOT = Path('/home/ubuntu/sepcung/03.Brute_Force_vs_Recon')
TII = Path('/data/Data_download_converted/TII-SSRC-23/extracted/pcap')
OUT = ROOT / 'Data_after_processing'
PKT_DIR = OUT / 'attack_packets'
TII_BEN_DIR = OUT / 'benign_tii'
REPORT_DIR = OUT / 'reports'
TII_BEN_DIR.mkdir(parents=True, exist_ok=True)

TII_ATT_IP, TII_ATT_MAC = '192.168.1.70', '00:0e:c6:7c:5a:5c'
TII_VIC_IP, TII_VIC_MAC = '192.168.1.90', '00:0e:c6:33:aa:b7'
ATTACKER_IP, ATTACKER_MAC = '192.168.1.236', '50:bb:b5:fe:d8:b8'  # khớp 01/03b
TARGET_IP, TARGET_MAC = '192.168.1.182', 'e4:5f:01:a3:c6:0d'

DOS_FILES = {'dos_icmp': 'icmp_dos.pcap', 'dos_syn': 'syn_tcp_dos.pcap', 'dos_udp': 'udp_dos.pcap'}
INCLUDE_TCP_FLAG_FLOODS = True  # thêm ACK/FIN/RST flood (TII cùng định dạng); False = chỉ ICMP/SYN/UDP như v3
if INCLUDE_TCP_FLAG_FLOODS:
    DOS_FILES |= {'dos_ack': 'ack_tcp_dos.pcap', 'dos_fin': 'fin_tcp_dos.pcap', 'dos_rst': 'rst_tcp_dos.pcap'}
BENIGN_FILES = {'video_http': 'video/http.pcap', 'video_rtp': 'video/rtp.pcap', 'video_udp': 'video/udp.pcap',
                'audio': 'audio/audio.pcap', 'text': 'text/text.pcap', 'background': 'background/background.pcap'}

BURST_GAP_S = 3.0     # gói .70→.90 cách gói trước > 3 s -> đợt mới (giữa các đợt TII nghỉ ~5 s)
MIN_DOS_PPS = 100     # đợt chậm hơn -> bỏ
SESSION_GAP_S = 60    # đợt giữ lại cách nhau ≤ 60 s -> cùng phiên (task)
RESP_TAIL_S = 1.0     # gói trả lời của nạn nhân tới sau gói tấn công cuối của đợt
SPLIT_SHARE = {'val': 0.15, 'test': 0.15}
N_JOBS = 6            # mỗi file DoS ~2–4 triệu gói, ~4 GB RAM khi đọc

## 1. Đọc pcap (cùng trường/cột với notebook 01, đọc luồng bằng `read_csv` để đủ RAM cho file vài GB)

In [2]:
FIELDS = ['frame.time_epoch', 'frame.len', 'eth.src', 'eth.dst', 'ip.src', 'ip.dst', 'ip.proto', 'ip.ttl',
          'tcp.srcport', 'tcp.dstport', 'tcp.flags', 'tcp.len', 'udp.srcport', 'udp.dstport', 'icmp.type',
          'arp.opcode', 'arp.src.hw_mac', 'arp.src.proto_ipv4', 'arp.dst.proto_ipv4']
COLS = ['ts', 'len', 'eth_src', 'eth_dst', 'ip_src', 'ip_dst', 'ip_proto', 'ttl',
        'tcp_sport', 'tcp_dport', 'tcp_flags', 'tcp_len', 'udp_sport', 'udp_dport', 'icmp_type',
        'arp_op', 'arp_sha', 'arp_spa', 'arp_tpa']
INT_COLS = ['len', 'ip_proto', 'ttl', 'tcp_sport', 'tcp_dport', 'tcp_len', 'udp_sport', 'udp_dport', 'icmp_type', 'arp_op']


def read_pcap(pcap):
    cmd = ['tshark', '-r', str(pcap), '-n', '-T', 'fields', '-E', 'separator=\t', '-E', 'occurrence=f']
    for f in FIELDS:
        cmd += ['-e', f]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True) as p:
        df = pd.read_csv(p.stdout, sep='\t', header=None, names=COLS, dtype=str, keep_default_na=False,
                         quoting=csv.QUOTE_NONE)
    assert p.returncode == 0, f'tshark lỗi với {pcap}'
    df['ts'] = pd.to_numeric(df['ts'], errors='coerce')
    for c in INT_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int32')
    df['tcp_flags'] = df['tcp_flags'].map(lambda x: int(x, 16) if x else pd.NA).astype('Int32')
    # ip.proto lấy lớp ngoài: cổng TCP/UDP bên trong ICMP error không được tính là gói TCP/UDP
    is_icmp = df['ip_proto'] == 1
    df.loc[is_icmp, ['tcp_sport', 'tcp_dport', 'tcp_flags', 'tcp_len', 'udp_sport', 'udp_dport']] = pd.NA
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].str.lower()
    return df.sort_values('ts', kind='stable').reset_index(drop=True)


def _remap(df, ip_old, mac_old, ip_new, mac_new):  # như 03b
    for c in ['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa']:
        df[c] = df[c].mask(df[c] == ip_old, ip_new)
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].mask(df[c] == mac_old, mac_new)
    return df

## 2. DoS: đợt → lọc tốc độ → phiên → gán nhãn, đổi địa chỉ, lưu task

Một file TII là một loại DoS; mỗi phiên lưu thành `task_XXXX.parquet` (đánh số theo thời gian).

In [3]:
def assign_split(n):
    # phiên đã sắp theo thời gian: sớm -> train, muộn -> val rồi test
    if n == 1:
        return ['train']
    if n == 2:
        return ['train', 'test']
    n_va, n_te = (max(1, round(SPLIT_SHARE[s] * n)) for s in ['val', 'test'])
    return ['train'] * (n - n_va - n_te) + ['val'] * n_va + ['test'] * n_te


def process_dos(scen, fname):
    pk = read_pcap(TII / 'malicious' / 'dos' / fname)
    src, dst = pk['ip_src'], pk['ip_dst']
    fwd = ((src == TII_ATT_IP) & (dst == TII_VIC_IP)).to_numpy()
    pair = fwd | ((src == TII_VIC_IP) & (dst == TII_ATT_IP)).to_numpy()
    ts = pk['ts'].to_numpy()

    # đợt theo gói gửi .70 -> .90
    tf = ts[fwd]
    bid = np.r_[0, np.cumsum(np.diff(tf) > BURST_GAP_S)]
    b = pd.DataFrame({'t0': tf, 't1': tf, 'n_fwd': 1}).groupby(bid).agg(t0=('t0', 'min'), t1=('t1', 'max'),
                                                                         n_fwd=('n_fwd', 'sum'))
    b['dur_s'] = b['t1'] - b['t0']
    b['pps'] = b['n_fwd'] / b['dur_s'].clip(lower=1.0)
    b['keep'] = b['pps'] >= MIN_DOS_PPS
    k = b[b['keep']]
    sess = np.r_[0, np.cumsum(k['t0'].to_numpy()[1:] - k['t1'].to_numpy()[:-1] > SESSION_GAP_S)]
    b['session'] = -1
    b.loc[k.index, 'session'] = sess

    # gán mỗi gói của cặp vào đợt (kể cả trả lời muộn RESP_TAIL_S)
    i = np.searchsorted(b['t0'].to_numpy(), ts, side='right') - 1
    ok = i >= 0
    i = np.maximum(i, 0)
    in_burst = ok & (ts <= b['t1'].to_numpy()[i] + RESP_TAIL_S) & b['keep'].to_numpy()[i]
    sess_of = np.where(in_burst, b['session'].to_numpy()[i], -1)
    attack = pair & in_burst
    is_arp = pk['arp_op'].notna().to_numpy()
    tii_arp = is_arp & (pk['arp_spa'].isin([TII_ATT_IP, TII_VIC_IP]) | pk['arp_tpa'].isin([TII_ATT_IP, TII_VIC_IP])).to_numpy()

    out_dir = PKT_DIR / scen
    out_dir.mkdir(parents=True, exist_ok=True)
    tasks = []
    n_sess = int(sess.max()) + 1 if len(sess) else 0
    splits = assign_split(n_sess)
    for s in range(n_sess):
        bs = b[b['session'] == s]
        t0, t1 = bs['t0'].min(), bs['t1'].max() + RESP_TAIL_S
        m_att = attack & (sess_of == s)
        m_aux = tii_arp & (ts >= t0 - BURST_GAP_S) & (ts <= t1)
        d = pk[m_att | m_aux].copy()
        d['label'] = np.where(m_att[m_att | m_aux], 'attack', 'aux')
        d = _remap(_remap(d, TII_ATT_IP, TII_ATT_MAC, ATTACKER_IP, ATTACKER_MAC),
                   TII_VIC_IP, TII_VIC_MAC, TARGET_IP, TARGET_MAC)
        task_id = s + 1
        path = out_dir / f'task_{task_id:04d}.parquet'
        d.to_parquet(path, index=False)
        a = d[d['label'] == 'attack']
        a_fwd = (a['ip_src'] == ATTACKER_IP).sum()
        tasks.append({'scenario': scen, 'rule': 'dos', 'class': 'dos', 'task_id': task_id, 'tool': 'hping3 (TII-SSRC-23)',
                      'source_pcap': fname, 'split': splits[s], 'bursts': len(bs),
                      'pps_min': bs['pps'].min(), 'pps_max': bs['pps'].max(),
                      'attack_t0': a['ts'].min(), 'attack_t1': a['ts'].max(),
                      'attack_duration_s': a['ts'].max() - a['ts'].min(),
                      'n_attack': len(a), 'n_attack_fwd': int(a_fwd), 'n_attack_rev': int(len(a) - a_fwd),
                      'n_aux': int((d['label'] == 'aux').sum()), 'parquet': str(path), 'usable': True})
    b.insert(0, 'scenario', scen)
    b['split'] = b['session'].map(dict(enumerate(splits))).fillna('')
    return b.reset_index(names='burst'), pd.DataFrame(tasks), {
        'scenario': scen, 'pcap_pkts': len(pk), 'pair_pkts': int(pair.sum()), 'attack_pkts': int(attack.sum()),
        'dropped_low_rate_pkts': int((pair & ~in_burst).sum()),
        'other_pkts': int((~pair & ~tii_arp).sum())}


with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    dos_res = list(ex.map(process_dos, DOS_FILES, DOS_FILES.values()))
bursts = pd.concat([r[0] for r in dos_res], ignore_index=True)
dos_task = pd.concat([r[1] for r in dos_res], ignore_index=True)
bursts.to_csv(REPORT_DIR / '01c_dos_bursts.csv', index=False)
dos_task.to_csv(REPORT_DIR / '01c_dos_task_inventory.csv', index=False)
pd.DataFrame([r[2] for r in dos_res]).set_index('scenario')

,pcap_pkts,pair_pkts,attack_pkts,dropped_low_rate_pkts,other_pkts
scenario,,,,,
dos_icmp,1650100,1650100,1636941,13159,0
dos_syn,3839202,3839202,3833209,5993,0
dos_udp,1909835,1909835,1900569,9266,0
dos_ack,4075758,4075758,4069784,5974,0
dos_fin,3670568,3670568,3663348,7220,0
dos_rst,3835959,3835959,3831760,4199,0


In [4]:
# Đợt bị bỏ vì chậm và phân bố tốc độ đợt giữ lại
bursts.assign(rate=pd.cut(bursts['pps'], [0, MIN_DOS_PPS, 1000, 5000, np.inf],
                          labels=[f'<{MIN_DOS_PPS} (bỏ)', f'{MIN_DOS_PPS}–1k', '1k–5k', '>5k'])) \
    .pivot_table(index='scenario', columns='rate', values='n_fwd', aggfunc=['size', 'sum'], observed=False, fill_value=0)

size                        sum                        
rate     <100 (bỏ) 100–1k 1k–5k >5k <100 (bỏ) 100–1k   1k–5k      >5k
scenario                                                             
dos_ack         36     12    30  54      4200  25775  621974  2628508
dos_fin         35     12    31  46      4097  25768  654082  2412412
dos_icmp         6      2     6  10      6580  40413  213696   812439
dos_rst         36     12    29  56      4199  25735  664466  3119069
dos_syn         37     12    33  51      4135  25728  720973  2379078
dos_udp         12      4     9  23      9266  56845  267933  1575791

In [5]:
# Task (phiên) theo split
dos_task.groupby(['scenario', 'split']).agg(
    tasks=('task_id', 'size'), bursts=('bursts', 'sum'), attack_pkts=('n_attack', 'sum'),
    duration_s=('attack_duration_s', 'sum'), pps_min=('pps_min', 'min'), pps_max=('pps_max', 'max')).round(1)

tasks  bursts  attack_pkts  duration_s  pps_min  pps_max
scenario split                                                          
dos_ack  test       2      12       322106       120.7    369.6   7191.1
         train     12      72      3243191       724.3    369.1  17714.4
         val        2      12       504487       120.9   3636.1   8007.5
dos_fin  test       2      12       324736       120.8    369.2   6947.4
         train     11      65      2826769       653.8    369.3  18573.9
         val        2      12       511843       121.2   3171.0   9443.6
dos_icmp test       1       1        73275       100.0    367.2    367.2
         train      4      13      1290922       175.7    369.3  13051.0
         val        1       4       272744        55.3   3320.9   6353.5
dos_rst  test       2      12       278701       120.5    369.1   8254.8
         train     12      73      3068604       733.6    368.6  19086.0
         val        2      12       484455       120.7   3567.6   9877.1
dos_syn  test       2      12       321927       120.8    368.7   7691.7
         train     12      72      3028146       725.9    367.1  14771.0
         val        2      12       483136       121.0   3601.7   7808.6
dos_udp  test       2      10       405180       240.4    366.9   8487.5
         train      8      22      1407822       226.8    368.9  16853.1
         val        2       4        87567        40.4   2507.5   3163.8

## 3. Benign TII: cắt liên tục theo thời gian 70/15/15 → train/val/test

Cắt theo số gói tích luỹ (không theo giờ — các file kéo dài ~1 ngày nhưng phần lớn thời gian im lặng).

In [6]:
def process_benign(name, rel):
    pk = read_pcap(TII / 'benign' / rel)
    pk['label'] = ''
    q = np.arange(len(pk)) / max(len(pk), 1)
    cut = 1 - SPLIT_SHARE['val'] - SPLIT_SHARE['test']
    sp = np.where(q < cut, 'train', np.where(q < cut + SPLIT_SHARE['val'], 'val', 'test'))
    rows = []
    for s in ['train', 'val', 'test']:
        d = pk[sp == s]
        path = TII_BEN_DIR / f'{name}_{s}.parquet'
        d.to_parquet(path, index=False)
        rows.append({'name': name, 'split': s, 'pkts': len(d), 't0': d['ts'].min(), 't1': d['ts'].max(),
                     'hours': (d['ts'].max() - d['ts'].min()) / 3600 if len(d) else 0,
                     'top_pair': (d['ip_src'] + '>' + d['ip_dst']).value_counts().index[0] if len(d) else '',
                     'parquet': str(path)})
    return rows


with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    tii_parts = pd.DataFrame([r for rows in ex.map(process_benign, BENIGN_FILES, BENIGN_FILES.values()) for r in rows])
tii_parts.to_csv(REPORT_DIR / '01c_tii_benign_parts.csv', index=False)
tii_parts.drop(columns='parquet').round(2)

,name,split,pkts,t0,t1,hours,top_pair
0,video_http,train,565188,2517.66,85168.57,22.96,192.168.1.70>192.168.1.90
1,video_http,val,121112,85168.57,85594.17,0.12,192.168.1.70>192.168.1.90
2,video_http,test,121111,85594.17,86018.46,0.12,192.168.1.70>192.168.1.90
3,video_rtp,train,403136,4091.96,86325.74,22.84,192.168.1.70>192.168.1.90
4,video_rtp,val,86386,86325.74,86898.00,0.16,192.168.1.70>192.168.1.90
5,video_rtp,test,86386,86898.00,87390.26,0.14,192.168.1.70>192.168.1.90
6,video_udp,train,388766,5709.78,18790.99,3.63,192.168.1.70>192.168.1.90
7,video_udp,val,83307,18791.00,88464.53,19.35,192.168.1.70>192.168.1.90
8,video_udp,test,83307,88464.53,89182.65,0.20,192.168.1.70>192.168.1.90
9,audio,train,524732,13306.54,90944.78,21.57,192.168.1.3>192.168.1.90


In [7]:
# Spot-check: chữ ký gói attack theo loại (proto/cờ TCP/ICMP) và chiều — như mục spot-check của 01
def sig(df):
    p = df['ip_proto'].astype('string').fillna('arp')
    f = df['tcp_flags'].map(lambda x: '' if pd.isna(x) else f'0x{int(x):02x}')
    i = df['icmp_type'].astype('string').fillna('')
    return p + '/' + f + '/' + i


spot = []
for r in dos_task.groupby('scenario').head(2).to_dict('records'):
    d = pd.read_parquet(r['parquet'])
    d = d[d['label'] == 'attack']
    d = d.assign(dir=np.where(d['ip_src'] == ATTACKER_IP, 'from_attacker', 'to_attacker'), s=sig(d))
    spot.append(d.groupby(['dir', 's']).size().rename('n').reset_index().assign(scenario=r['scenario']))
spot = pd.concat(spot).groupby(['scenario', 'dir', 's'])['n'].sum().reset_index()
spot.columns = ['scenario', 'dir', 'proto/flags/icmp', 'n']
spot.sort_values(['scenario', 'n'], ascending=[True, False]).groupby('scenario').head(5)

,scenario,dir,proto/flags/icmp,n
1,dos_ack,from_attacker,6/0x10/,770068
2,dos_ack,to_attacker,6/0x04/,326084
0,dos_ack,from_attacker,6/0x01/,68071
3,dos_ack,to_attacker,6/0x14/,25993
4,dos_fin,from_attacker,6/0x01/,910567
5,dos_fin,to_attacker,6/0x14/,265035
7,dos_icmp,from_attacker,1//8,209690
9,dos_icmp,to_attacker,1//0,118883
6,dos_icmp,from_attacker,1//,108483
8,dos_icmp,to_attacker,1//,25023
